# Trying to fix Spatial

In [1]:
import sys
import os

project_path = "/home/sagemaker-user/gbm_hackathon"
if project_path not in sys.path:
    sys.path.append(project_path)
    print(sys.path)

['/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python310.zip', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/lib-dynload', '', '/home/sagemaker-user/.conda/envs/gbmhackathon/lib/python3.10/site-packages', '/home/sagemaker-user/gbm_hackathon']


In [2]:
%load_ext autoreload
%autoreload 2
from gbmhackathon.s3_loader import load_s3

In [3]:
BUCKET_MOSAIC = "ABSTRA_DATASET_03bb30aa_16ed_4b89_913e_fe009db2aabd"
BUCKET_PROJECT = "ABSTRA_PROJECT_STORAGE_BUCKET"

def fetch_path(env_var_name):
    return os.path.expandvars(f"${env_var_name}")

S3_PATH_CONNECTIVITIES_EMB = fetch_path(BUCKET_PROJECT) + "embedding_V1/2025-04-26_16-02_connectivities_spatial_V1.pkl"
S3_PATH_NOVAE_EMB = fetch_path(BUCKET_PROJECT) + "embedding_V1/2025-03-23_18-32_spatial_emb_V1.pkl"

In [4]:
connectivities = load_s3(S3_PATH_CONNECTIVITIES_EMB)
novae_embs = load_s3(S3_PATH_NOVAE_EMB)

In [5]:
for val in connectivities.values():
    print(val.size())

torch.Size([2, 8286])
torch.Size([2, 25976])
torch.Size([2, 16132])
torch.Size([2, 12658])
torch.Size([2, 20752])
torch.Size([2, 28300])
torch.Size([2, 28096])
torch.Size([2, 27464])
torch.Size([2, 26906])
torch.Size([2, 15508])
torch.Size([2, 17698])
torch.Size([2, 18724])
torch.Size([2, 20028])
torch.Size([2, 8426])
torch.Size([2, 14258])
torch.Size([2, 18282])
torch.Size([2, 25698])
torch.Size([2, 20728])
torch.Size([2, 15184])
torch.Size([2, 22156])
torch.Size([2, 23518])
torch.Size([2, 12514])
torch.Size([2, 27068])
torch.Size([2, 26598])
torch.Size([2, 24796])
torch.Size([2, 26166])
torch.Size([2, 28266])
torch.Size([2, 16344])
torch.Size([2, 26454])
torch.Size([2, 26482])
torch.Size([2, 13252])
torch.Size([2, 22282])
torch.Size([2, 27370])
torch.Size([2, 24486])
torch.Size([2, 27252])
torch.Size([2, 21432])
torch.Size([2, 26324])
torch.Size([2, 15836])
torch.Size([2, 26994])
torch.Size([2, 12394])
torch.Size([2, 21324])
torch.Size([2, 25494])
torch.Size([2, 28552])
torch.Size([2

In [6]:
for val in novae_embs.values():
    print(val.size())

torch.Size([1462, 64])
torch.Size([4467, 64])
torch.Size([2876, 64])
torch.Size([2237, 64])
torch.Size([3589, 64])
torch.Size([4832, 64])
torch.Size([4773, 64])
torch.Size([4672, 64])
torch.Size([4583, 64])
torch.Size([2771, 64])
torch.Size([3166, 64])
torch.Size([3209, 64])
torch.Size([3500, 64])
torch.Size([1472, 64])
torch.Size([2464, 64])
torch.Size([3233, 64])
torch.Size([4401, 64])
torch.Size([3604, 64])
torch.Size([2712, 64])
torch.Size([3846, 64])
torch.Size([4095, 64])
torch.Size([2236, 64])
torch.Size([4663, 64])
torch.Size([4576, 64])
torch.Size([4240, 64])
torch.Size([4480, 64])
torch.Size([4829, 64])
torch.Size([2900, 64])
torch.Size([4502, 64])
torch.Size([4522, 64])
torch.Size([2391, 64])
torch.Size([3856, 64])
torch.Size([4672, 64])
torch.Size([4175, 64])
torch.Size([4692, 64])
torch.Size([3808, 64])
torch.Size([4513, 64])
torch.Size([2732, 64])
torch.Size([4656, 64])
torch.Size([2174, 64])
torch.Size([3708, 64])
torch.Size([4395, 64])
torch.Size([4866, 64])
torch.Size(

# **Connectivites is not as expected => We'll fall back to simply using the Average of NOVAE Embeddings**
> We need to fix the collate functions

In [7]:
from gbmhackathon.training.predictive import PredictiveLearningDataset, collate_predictive
from gbmhackathon.training.patientwise import PatientLearningDataset, collate_patient_wise
import torch
from torch.utils.data import DataLoader

In [8]:
name_emb_dict = {"hne":"embeddings_HnE_OptimusH0.pkl",
"spatial":"2025-03-23_18-32_spatial_emb_V1.pkl",
"clinical":"2025-05-25_14-36_new_clinical_emb_V1.pkl",
"wes":"2025-04-05_13-40_wes_emb_V1.pkl",
"bulk":"2025-05-03_10-15_bulk_emb_V1.pkl",
"scRNA":"2025-05-04_02-35_scRNA_emb_V1.pkl"}
pkl_storage_folder = "embedding_V1"

In [9]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dataset = PredictiveLearningDataset(name_emb_dict, pkl_storage_folder, device=device, dropout=1.0)
print(f"Dataset size: {len(dataset)}")
BATCH_SIZE = 582
dataloader = DataLoader(dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_predictive, generator=torch.Generator(device=dataset.device))

cas 2 : device reconnu : cpu 
Using device: cpu
By keeping 100.00% of dropout augmented samples we went from:
628 dropout samples (84.64% dropout in dataset) -- to --> 628 dropout samples (84.64% dropout in dataset)
Dataset size: 742


In [10]:
next(iter(dataloader))[2]['spatial'].size()

torch.Size([582, 64])

In [11]:
dataset = PatientLearningDataset(name_emb_dict, pkl_storage_folder, device=device, dropout=1.0)
print(f"Dataset size: {len(dataset)}")
BATCH_SIZE = 582
dataloader = DataLoader(dataset, BATCH_SIZE, shuffle=True, collate_fn=collate_patient_wise, generator=torch.Generator(device=dataset.device))

Using device : cpu
By keeping 100.00% of dropout augmented samples we went from:
628 dropout samples (84.64% dropout in dataset) -- to --> 628 dropout samples (84.64% dropout in dataset)
Dataset size: 742


In [12]:
next(iter(dataloader))[2]['spatial'].size()

torch.Size([582, 64])

# Fixed

In [13]:
from gbmhackathon.models.dim_reduction import *

In [14]:
batch_all = [dataset.__getitem__(idx) for idx in dataset.ind2patient if 'd' not in dataset.ind2patient[idx]]
batch_all = collate_patient_wise(batch_all)

In [36]:
dr = TSNE()

In [37]:
dr(batch_all[2]['spatial']).size()

Size after compression: 21


ValueError: 'n_components' should be inferior to 4 for the barnes_hut algorithm as it relies on quad-tree or oct-tree.